# 00 — Datenaufbereitung: 5 Datenvarianten

Dieses Notebook bereitet die Rohdaten fuer die Experimentreihe auf. Es erzeugt **5 Datenvarianten** (D1–D5) aus den CSV-Dateien der 18 Gemeinden.

## Ueberblick

| Variante | Gemeinden | Zweck |
|----------|-----------|-------|
| D1_single | GM0047 | Baseline: Einzelne Gemeinde |
| D2_multi4 | GM0047 + 3 weitere | Multi-Gemeinde |
| D3_mittel6 | 6 diverse Gemeinden | Mittlere Vielfalt |
| D4_gross10 | 10 Gemeinden | Maximale Trainingsdaten |
| D5_fremd | D3 ohne GM0047 | Haertester Generalisierungstest |

## Pipeline
1. **Laden**: travel_times + dwell_times CSVs pro Gemeinde
2. **Bereinigen**: Ausreisser bei Fahrzeit (0–600s), Haltezeit (>300s), Geschwindigkeit (<100 km/h)
3. **Feature Engineering**: 20 Features (zyklisch, one-hot, Haversine, Label-Encoding, Lag)
4. **Split**: Chronologisch 70% Train / 15% Validation / 15% Test

**Laufzeit**: ca. 8 Minuten

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

SCRIPT_START = time.time()

## Pfade und Konfiguration

Die Pfade sind relativ zum Notebook-Ordner aufgebaut:
- `DATEN_DIR`: Rohdaten (Daten/travel_times/, Daten/dwell_times/)
- `DATA_DIR`: Ausgabe der aufbereiteten Daten (data/D1_single/, etc.)
- `OUTPUT_DIR`: Zusammenfassungen und Timing-Dateien

In [ ]:
# Pfade: Notebook liegt in skripte/notebooks/,
# BASE_DIR zeigt auf das Projekt-Root (2 Ebenen hoch)
BASE_DIR = Path(".").resolve().parent.parent  # -> Projekt-Root
DATEN_DIR = BASE_DIR / "Daten"
DATA_DIR = Path(".").resolve().parent / "data"
OUTPUT_DIR = Path(".").resolve().parent / "ergebnisse"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Rohdaten:  {DATEN_DIR}")
print(f"Ausgabe:   {DATA_DIR}")

## Datenvarianten-Definition

Jede Variante ist eine Liste von Gemeinde-IDs (z.B. `GM0047`). Die Varianten sind so gewaehlt, dass sie verschiedene Aspekte der Generalisierung testen:

- **D1–D4**: Steigende Anzahl an Gemeinden, immer mit GM0047
- **D5**: Ohne GM0047 → testet, ob ein Modell auf voellig unbekannten Gemeinden funktioniert

In [ ]:
VARIANTEN = {
    "D1_single":  ["GM0047"],
    "D2_multi4":  ["GM0047", "GM0059", "GM0281", "GM0590"],
    "D3_mittel6": ["GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681"],
    "D4_gross10": ["GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681",
                   "GM0281", "GM1950", "GM1969", "GM1930"],
    "D5_fremd":   ["GM0312", "GM0590", "GM0546", "GM0629", "GM1681"],
}

for name, gms in VARIANTEN.items():
    print(f"{name:15s}: {len(gms)} Gemeinden — {gms}")

## Feature-Definition (20 Features)

Die Features decken verschiedene Aspekte ab:

| Kategorie | Features | Beschreibung |
|-----------|----------|--------------|
| Zeitlich (zyklisch) | hour_sin, hour_cos, month_sin, month_cos | Vermeidet Sprung bei 23→0 Uhr bzw. Dez→Jan |
| Zeitlich (diskret) | weekday_1..6, is_weekend, is_rush_hour | One-Hot mit Montag als Referenz |
| Raeumlich | segment_dist_m | Haversine-Distanz zwischen Haltestellen |
| Kategorisch | from_stop_enc, to_stop_enc, route_enc, line_enc | Label-Encoding (0..n-1) |
| Betrieblich | dwell_time, seg_position | Haltestellenaufenthalt, Segment-Position im Trip |
| Lag | travel_time_prev | Fahrzeit des vorherigen Segments |

In [ ]:
FEATURE_COLS = (
    ["hour_sin", "hour_cos"]
    + [f"weekday_{d}" for d in range(1, 7)]
    + ["is_weekend", "is_rush_hour"]
    + ["month_sin", "month_cos"]
    + ["segment_dist_m", "from_stop_enc", "to_stop_enc", "route_enc", "line_enc"]
    + ["dwell_time", "seg_position"]
    + ["travel_time_prev"]
)
TARGET = "travel_time"

print(f"Anzahl Features: {len(FEATURE_COLS)}")
print(f"Target: {TARGET}")
print(f"\nFeature-Liste:")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {f}")

## Hilfsfunktionen

### WKT-POINT-Parser
Die Geometriedaten liegen als WKT-Strings vor (z.B. `POINT(11.12345 48.67890)`). Diese Funktion extrahiert Longitude und Latitude als NumPy-Arrays.

### Haversine-Formel
Berechnet die Grosskreisdistanz (Luftlinie) zwischen zwei Koordinatenpaaren in Metern. Verwendet den Erdradius R = 6.371 km. Die Berechnung ist vektorisiert fuer ganze DataFrames.

In [ ]:
def parse_points_vectorized(series):
    """Extrahiert Lon/Lat aus WKT-POINT-Strings wie 'POINT(11.123 48.678)'."""
    extracted = series.str.extract(r"POINT\(([^ ]+) ([^ ]+)\)")
    return extracted[0].astype(float).values, extracted[1].astype(float).values


def haversine_vec(lon1, lat1, lon2, lat2):
    """Grosskreisdistanz in Metern zwischen zwei Koordinatenpaaren (vektorisiert)."""
    R = 6_371_000  # Erdradius in Metern
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

## Daten laden und bereinigen

Die Funktion `load_and_clean()` fuehrt fuer jede Gemeinde folgende Schritte aus:

1. **CSV laden**: travel_times und dwell_times
2. **Dauern berechnen**: `travel_time = to_time - from_time` (in Sekunden)
3. **Merge**: dwell_time per Left-Join an travel_times (ueber date, trip, route, from_stop)
4. **Ausreisser entfernen**:
   - `travel_time`: Nur 0–600 Sekunden (max. 10 Min. pro Segment)
   - `dwell_time > 300s`: Auf NaN setzen (unplausibel langer Halt)
   - Geschwindigkeit > 100 km/h: Entfernen (physikalisch unmoeglich fuer Busse)
5. **Haversine-Distanz**: Luftlinie zwischen Start- und Zielhaltestelle

In [ ]:
def load_and_clean(name: str) -> pd.DataFrame:
    """Laedt eine Gemeinde, merged dwell_times, bereinigt Ausreisser."""
    print(f"    [{name}] Laden...", end=" ", flush=True)
    df_tt = pd.read_csv(DATEN_DIR / "travel_times" / f"{name}.csv")
    df_dt = pd.read_csv(DATEN_DIR / "dwell_times" / f"{name}.csv")

    # Zeitstempel parsen und Dauern in Sekunden berechnen
    df_tt["from_time"] = pd.to_datetime(df_tt["from_time"])
    df_tt["to_time"] = pd.to_datetime(df_tt["to_time"])
    df_tt["travel_time"] = (df_tt["to_time"] - df_tt["from_time"]).dt.total_seconds()

    df_dt["from_time"] = pd.to_datetime(df_dt["from_time"])
    df_dt["to_time"] = pd.to_datetime(df_dt["to_time"])
    df_dt["dwell_time"] = (df_dt["to_time"] - df_dt["from_time"]).dt.total_seconds()

    # Datentyp-Fix: route ist in dwell_times manchmal float64
    df_tt["route"] = df_tt["route"].astype("int64")
    df_dt["route"] = df_dt["route"].astype("int64")

    # Left-Join: Haltestellenaufenthalt an Fahrzeitdaten
    df = pd.merge(
        df_tt,
        df_dt[["date", "trip", "route", "stop", "dwell_time"]],
        how="left",
        left_on=["date", "trip", "route", "from_stop"],
        right_on=["date", "trip", "route", "stop"],
    ).drop(columns=["stop"])

    n_before = len(df)

    # Ausreisser-Filterung
    mask_tt = (df["travel_time"] > 0) & (df["travel_time"] <= 600)
    df.loc[df["dwell_time"] > 300, "dwell_time"] = np.nan

    lon_from, lat_from = parse_points_vectorized(df["from_geometry"])
    lon_to, lat_to = parse_points_vectorized(df["to_geometry"])
    df["segment_dist_m"] = haversine_vec(lon_from, lat_from, lon_to, lat_to)

    df["speed_kmh"] = (df["segment_dist_m"] / df["travel_time"].replace(0, np.nan)) * 3.6
    mask_speed = df["speed_kmh"].fillna(0) <= 100

    df = df[mask_tt & mask_speed].copy()
    df.drop(columns=["speed_kmh"], inplace=True)
    df["gemeinde"] = name

    print(f"{n_before:>9,} -> {len(df):>9,} ({n_before - len(df):,} entfernt)")
    return df

## Feature Engineering

Die Funktion `engineer_features()` erzeugt die 20 Features aus den bereinigten Rohdaten:

### Zyklische Kodierung
Stunden und Monate werden als sin/cos-Paare kodiert, damit das Modell versteht, dass z.B. 23 Uhr und 0 Uhr nahe beieinander liegen (und nicht 23 Einheiten auseinander).

### One-Hot-Encoding der Wochentage
Montag ist die **Referenzkategorie** (alle weekday_1..6 = 0). Dies vermeidet die "Dummy-Variable-Falle" bei linearen Modellen.

### Label-Encoding
Kategorische Variablen (Haltestellen, Routen, Linien) werden als Ganzzahlen kodiert. **Achtung**: Die Codes sind nur innerhalb einer Datenvariante konsistent!

### Lag-Feature
`travel_time_prev` ist die Fahrzeit des vorherigen Segments im selben Trip. Das erste Segment hat keinen Vorgaenger und erhaelt den Wert 0.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Erzeugt 20 Features aus den bereinigten Rohdaten."""
    df = df.copy()
    df["date_dt"] = pd.to_datetime(df["date"])
    hour = df["from_time"].dt.hour

    # Zyklische Stunden (sin/cos, Periode 24h)
    df["hour_sin"] = np.sin(hour * 2 * np.pi / 24)
    df["hour_cos"] = np.cos(hour * 2 * np.pi / 24)

    # Wochentag (Mo=Referenz)
    weekday = df["date_dt"].dt.dayofweek
    for d in range(1, 7):
        df[f"weekday_{d}"] = (weekday == d).astype(int)
    df["is_weekend"] = (weekday >= 5).astype(int)
    df["is_rush_hour"] = (((hour >= 7) & (hour <= 9)) | ((hour >= 15) & (hour <= 18))).astype(int)

    # Zyklische Monate (sin/cos, Periode 12)
    month = df["date_dt"].dt.month
    df["month_sin"] = np.sin(month * 2 * np.pi / 12)
    df["month_cos"] = np.cos(month * 2 * np.pi / 12)

    # Label-Encoding
    for col, new_col in [("from_stop", "from_stop_enc"), ("to_stop", "to_stop_enc"),
                          ("route", "route_enc"), ("line", "line_enc")]:
        df[new_col] = df[col].astype("category").cat.codes

    # Betriebliche Features
    df["dwell_time"] = df["dwell_time"].fillna(0.0)
    df = df.sort_values(["date", "trip", "from_time"])
    df["seg_position"] = df.groupby(["date", "trip", "gemeinde"]).cumcount()

    # Lag-Feature: Fahrzeit des vorherigen Segments
    df["travel_time_prev"] = df.groupby(["date", "trip", "gemeinde"])["travel_time"].shift(1)
    df["travel_time_prev"] = df["travel_time_prev"].fillna(0.0)

    return df

## Train/Val/Test-Split und Export

Der Split erfolgt **chronologisch** (nicht zufaellig!):
- Die ersten 70% der Daten (zeitlich) = **Training**
- Die naechsten 15% = **Validation** (fuer Early Stopping)
- Die letzten 15% = **Test** (finale Evaluation)

**Warum chronologisch?** Bei Zeitreihendaten wuerde ein zufaelliger Split dazu fuehren, dass das Modell "in die Zukunft schaut" (Data Leakage). Benachbarte Zeitpunkte sind stark korreliert.

In [ ]:
def split_and_save(df: pd.DataFrame, variant_name: str):
    """Zeitbasierter 70/15/15 Split + Export als CSV."""
    out_dir = DATA_DIR / variant_name
    out_dir.mkdir(parents=True, exist_ok=True)

    df_model = df[FEATURE_COLS + [TARGET, "date_dt", "from_time"]].copy()
    df_model = df_model.sort_values(["date_dt", "from_time"]).reset_index(drop=True)

    # Chronologischer Split
    n = len(df_model)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    df_train = df_model.iloc[:n_train]
    df_val = df_model.iloc[n_train:n_train + n_val]
    df_test = df_model.iloc[n_train + n_val:]

    # Nur Features + Target exportieren
    drop_cols = ["date_dt", "from_time"]
    df_train.drop(columns=drop_cols).to_csv(out_dir / "train.csv", index=False)
    df_val.drop(columns=drop_cols).to_csv(out_dir / "val.csv", index=False)
    df_test.drop(columns=drop_cols).to_csv(out_dir / "test.csv", index=False)

    # Label-Mappings (Code -> Originalname)
    label_maps = {}
    for col in ["from_stop", "to_stop", "route", "line"]:
        if col in df.columns:
            cat = df[col].astype("category")
            mapping = dict(enumerate(cat.cat.categories))
            label_maps[col] = mapping
    pd.to_pickle(label_maps, out_dir / "label_mappings.pkl")

    print(f"    Train: {len(df_train):>10,}  Val: {len(df_val):>8,}  Test: {len(df_test):>8,}")
    for f in ["train.csv", "val.csv", "test.csv"]:
        size = (out_dir / f).stat().st_size / 1024 / 1024
        print(f"    {f}: {size:.1f} MB")

    return len(df_train), len(df_val), len(df_test)

## Gemeinden laden (mit Cache)

Jede Gemeinde wird nur **einmal** geladen und bereinigt. Da Gemeinden in mehreren Varianten vorkommen (z.B. GM0047 in D1–D4), spart der Cache erheblich Zeit.

In [ ]:
print("=" * 70)
print("Datenaufbereitung — 5 Varianten")
print("=" * 70)

# Alle benoetigten Gemeinden ermitteln
alle_gemeinden = sorted(set(gm for gms in VARIANTEN.values() for gm in gms))
print(f"\nBenoetigte Gemeinden: {len(alle_gemeinden)}")

# Cache: Jede Gemeinde nur einmal laden
cache = {}
for gm in alle_gemeinden:
    cache[gm] = load_and_clean(gm)

print(f"\nAlle Gemeinden geladen. Cache: {len(cache)} DataFrames")

## Varianten erzeugen

Fuer jede Variante:
1. Relevante Gemeinden aus dem Cache zusammenfuehren
2. Features berechnen
3. Chronologisch splitten und als CSV exportieren

In [ ]:
summary = []

for variant_name, gemeinden in VARIANTEN.items():
    t0 = time.time()
    print(f"\n{'='*70}")
    print(f"  {variant_name}: {gemeinden}")
    print(f"{'='*70}")

    # Gemeinden zusammenfuehren
    dfs = [cache[gm] for gm in gemeinden]
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Roh-Segmente: {len(df):,}")

    # Feature Engineering
    df = engineer_features(df)

    # Statistiken
    n_stops = df["from_stop_enc"].nunique() + df["to_stop_enc"].nunique()
    n_routes = df["route_enc"].nunique()
    n_lines = df["line_enc"].nunique()
    print(f"  Features: {len(FEATURE_COLS)} | Stops: {df['from_stop_enc'].nunique()}+{df['to_stop_enc'].nunique()} | "
          f"Routen: {n_routes} | Linien: {n_lines}")

    # Split & Export
    n_tr, n_va, n_te = split_and_save(df, variant_name)

    elapsed = time.time() - t0
    summary.append({
        "variante": variant_name,
        "gemeinden": len(gemeinden),
        "segmente": len(df),
        "train": n_tr, "val": n_va, "test": n_te,
        "from_stops": df["from_stop_enc"].nunique(),
        "to_stops": df["to_stop_enc"].nunique(),
        "routen": n_routes,
        "linien": n_lines,
        "zeit_s": round(elapsed, 1),
    })
    print(f"  Dauer: {elapsed:.1f}s")

## Zusammenfassung

In [ ]:
sum_df = pd.DataFrame(summary)
sum_df.to_csv(OUTPUT_DIR / "datenvarianten_summary.csv", index=False)
sum_df

In [ ]:
elapsed_total = time.time() - SCRIPT_START
print(f"Gesamtlaufzeit: {elapsed_total:.1f}s")